# HW 2 — From Scratch: Build Your Own Neural Network
**Modern Computer Vision** — Technion, Spring 2026

**Due:** May 10, 2026

**Student ID:**

In [1]:
STUDENT_ID = "PARTIAL_204"

---
## Overview

In this assignment you'll build a working neural network **from scratch** — no `torch.nn`, no `loss.backward()`, no autograd.
You'll use raw PyTorch tensors and implement everything yourself: forward passes, backward passes, gradient descent.
By the end, you'll train an MLP on MNIST and verify it matches PyTorch's built-in implementation.

The assignment has five parts:
1. **The backward tape** — how our mini-autograd records and replays operations
2. **Differentiable functions** — implement `linear`, `relu`, and `cross_entropy_loss`
3. **Layers and models** — wrap functions into reusable `Linear` layers, build classifiers
4. **SGD optimizer** — implement gradient descent
5. **Train on MNIST** — put it all together, then compare with PyTorch

In [2]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

---
## Part 1: The backward tape

When PyTorch computes `y = f(x)`, it secretly records the operation so it can later compute gradients.
We'll do the same thing, but explicitly.

Our mechanism is simple: we pass around a **list called `ctx`** (short for *context*).
Every time a differentiable function runs, it appends one entry to `ctx`:

```python
ctx.append([backward_fn, [output, input1, input2, ...]])
```

This says: *"to undo this operation, call `backward_fn` with these tensors."*

After the forward pass, `ctx` is a tape of everything that happened, in order.
To compute gradients, we just **pop from the end and replay backwards**:

```python
while ctx:
    backward_fn, args = ctx.pop()
    backward_fn(*args)
```

That's it — that's our autograd. Each `backward_fn` reads `output.grad` (the gradient flowing in)
and **accumulates** into `input.grad` (the gradient flowing out). Let's see it work.

### Helper: initialize gradients

Before backward functions can accumulate gradients with `+=`, the `.grad` attribute needs to exist.
This helper creates it (as zeros) if it doesn't exist yet.

In [3]:
def ensure_grad(tensors):
    """Create .grad (zeros) for each tensor that doesn't have one yet."""
    for t in tensors:
        if not hasattr(t, 'grad') or t.grad is None:
            t.grad = torch.zeros_like(t)


def backward(loss, ctx):
    """
    Run backward pass: set loss.grad = 1, then replay the tape in reverse.
    
    Args:
        loss: scalar tensor (the thing we're differentiating)
        ctx:  list of [backward_fn, args] built during forward pass
    """
    loss.grad = torch.ones_like(loss)
    while ctx:
        backward_fn, args = ctx.pop()
        ensure_grad(args)
        backward_fn(*args)

### Warm-up example: `mean`

Here's a complete example of a differentiable function following our pattern.
If $y = \text{mean}(x) = \frac{1}{n}\sum_i x_i$, then $\frac{\partial y}{\partial x_i} = \frac{1}{n}$,
and by the chain rule: $\nabla_x = \frac{\partial L}{\partial y} \cdot \frac{1}{n}$.

In [4]:
def mean_backward(y, x):
    """Backward for mean: gradient is y.grad / n, broadcast to x's shape."""
    x.grad += y.grad / x.numel()

def mean(x, ctx=None):
    """Forward: compute mean. If ctx is given, record for backward."""
    y = x.mean()
    if ctx is not None:
        ctx.append([mean_backward, [y, x]])
    return y


# Quick test: does our backward give the same gradients as PyTorch?
x = torch.randn(5, requires_grad=True)

# Our way
ctx = []
y = mean(x.detach().clone(), ctx=ctx)
backward(y, ctx)

# PyTorch's way
y_pt = x.mean()
y_pt.backward()

print(f"Our gradient:     {y.grad.item():.4f} → x.grad = {x.detach().clone().grad}")
print(f"PyTorch gradient: {y_pt.grad_fn} → x.grad = {x.grad}")
print("✓ Match!") if torch.allclose(torch.ones(5) / 5, x.grad) else print("✗ Mismatch")

Our gradient:     1.0000 → x.grad = None
PyTorch gradient: <MeanBackward0 object at 0x124a56800> → x.grad = tensor([0.2000, 0.2000, 0.2000, 0.2000, 0.2000])
✓ Match!


---
## Part 2: Differentiable functions

Now you'll implement the building blocks of a neural network.
Each function follows the same pattern as `mean` above:

- A **backward function** that reads `output.grad` and accumulates into `input.grad`
- A **forward function** that computes the output and (optionally) records to `ctx`

Gradients always **accumulate** (`+=`), never overwrite — this is important when
a tensor is used in multiple operations.

### 2.1 Linear (fully connected)

The linear function computes: $\quad y = x \, W^T + b$

where $x$ is $(B, D_{\text{in}})$, $W$ is $(D_{\text{out}}, D_{\text{in}})$, $b$ is $(D_{\text{out}},)$, and $y$ is $(B, D_{\text{out}})$.

The backward pass needs three gradients (all derived from the chain rule):

$$\nabla_x = \nabla_y \, W \qquad \nabla_W = \nabla_y^T \, x \qquad \nabla_b = \sum_{\text{batch}} \nabla_y$$

In [5]:
def linear_backward(y, x, w, b):
    x.grad += y.grad @ w
    w.grad += y.grad.T @ x
    b.grad += y.grad.sum(dim=0)


def linear(x, w, b, ctx=None):
    y = x @ w.T + b
    if ctx is not None:
        ctx.append([linear_backward, [y, x, w, b]])
    return y

In [ ]:
# --- Sanity check: compare with PyTorch ---
torch.manual_seed(42)
B, Din, Dout = 4, 3, 2
x_pt = torch.randn(B, Din, requires_grad=True)
w_pt = torch.randn(Dout, Din, requires_grad=True)
b_pt = torch.randn(Dout, requires_grad=True)

# PyTorch
y_pt = x_pt @ w_pt.T + b_pt
loss_pt = y_pt.mean()
loss_pt.backward()

# Ours
x_ours = x_pt.detach().clone()
w_ours = w_pt.detach().clone()
b_ours = b_pt.detach().clone()
ctx = []
y_ours = linear(x_ours, w_ours, b_ours, ctx=ctx)
loss_ours = mean(y_ours, ctx=ctx)
backward(loss_ours, ctx)

assert torch.allclose(y_ours, y_pt.detach(), atol=1e-6), "Forward mismatch!"
assert torch.allclose(x_ours.grad, x_pt.grad, atol=1e-6), "x.grad mismatch!"
assert torch.allclose(w_ours.grad, w_pt.grad, atol=1e-6), "w.grad mismatch!"
assert torch.allclose(b_ours.grad, b_pt.grad, atol=1e-6), "b.grad mismatch!"
print("✓ linear forward and backward match PyTorch!")

[Grand test: skipped execution of this cell for speed]


### 2.2 ReLU

$\text{ReLU}(x) = \max(x, 0)$

Backward: the gradient passes through where $x > 0$, and is zero where $x \leq 0$.

$$\nabla_x = \nabla_y \odot \mathbf{1}[x > 0]$$

In [6]:
def relu_backward(y, x):
    x.grad += y.grad * (x > 0).float()


def relu(x, ctx=None):
    y = x.clamp(min=0).clone()
    if ctx is not None:
        ctx.append([relu_backward, [y, x]])
    return y

In [ ]:
# --- Sanity check ---
x_pt = torch.randn(3, 4, requires_grad=True)
y_pt = torch.relu(x_pt)
y_pt.mean().backward()

x_ours = x_pt.detach().clone()
ctx = []
y_ours = relu(x_ours, ctx=ctx)
loss = mean(y_ours, ctx=ctx)
backward(loss, ctx)

assert torch.allclose(y_ours, y_pt.detach(), atol=1e-6), "Forward mismatch!"
assert torch.allclose(x_ours.grad, x_pt.grad, atol=1e-6), "Backward mismatch!"
print("✓ relu forward and backward match PyTorch!")

[Grand test: skipped execution of this cell for speed]


### 2.3 Cross-entropy loss

We combine softmax and cross-entropy into one function (like `F.cross_entropy` in PyTorch).
This is numerically more stable and simpler to implement than doing them separately.

Given logits $z$ (raw model outputs, shape $(B, C)$) and integer targets $t$ (shape $(B,)$):

$$\text{loss} = -\frac{1}{B} \sum_{i=1}^{B} \left[ z_{i, t_i} - \log \sum_c e^{z_{ic}} \right]$$

This is equivalent to: softmax the logits, take the log of the target class probability, negate, average over the batch.

The backward pass has a famously clean form. If $p = \text{softmax}(z)$:

$$\nabla_{z_{ic}} = \frac{1}{B} \left( p_{ic} - \mathbf{1}[c = t_i] \right)$$

That is: the gradient is just the softmax probabilities, minus 1 at the correct class, averaged over the batch.

In [7]:
def cross_entropy_loss_backward(loss, z, targets):
    """
    Backward for combined softmax + cross-entropy loss.
    
    Accumulate into z.grad: (softmax(z) - one_hot(targets)) / B, 
    scaled by loss.grad.
    
    Args:
        loss: scalar loss tensor
        z:    (B, C) logits
        targets: (B,) integer class labels
    """
    # YOUR CODE HERE
    raise NotImplementedError


def cross_entropy_loss(z, targets, ctx=None):
    """
    Forward: softmax cross-entropy loss.
    
    Args:
        z:       (B, C) raw logits (not probabilities!)
        targets: (B,) integer class labels in [0, C)
        ctx:     backward tape
    Returns:
        loss: scalar tensor
    
    Hint: for numerical stability, subtract max(z) per row before exp.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- Sanity check ---
torch.manual_seed(0)
z_pt = torch.randn(8, 5, requires_grad=True)
t = torch.randint(0, 5, (8,))

loss_pt = F.cross_entropy(z_pt, t)
loss_pt.backward()

z_ours = z_pt.detach().clone()
ctx = []
loss_ours = cross_entropy_loss(z_ours, t, ctx=ctx)
backward(loss_ours, ctx)

assert torch.allclose(loss_ours, loss_pt.detach(), atol=1e-5), \
    f"Loss mismatch: ours={loss_ours.item():.6f}, PT={loss_pt.item():.6f}"
assert torch.allclose(z_ours.grad, z_pt.grad, atol=1e-5), \
    f"Gradient mismatch! Max diff: {(z_ours.grad - z_pt.grad).abs().max():.6f}"
print(f"✓ cross_entropy_loss matches PyTorch! (loss = {loss_ours.item():.4f})")

[Grand test: skipped execution of this cell for speed]


---
## Part 3: Layers and models

Now we wrap our functions into reusable building blocks.

A **layer** holds learnable parameters (weights, biases) and delegates computation
to the functions you just wrote. A **model** chains layers together.

### Module base class (provided)

This is a minimal version of `torch.nn.Module`. It manages parameter collection
and device transfer — you don't need to modify it.

In [8]:
class Module:
    """Base class for all layers and models."""
    
    def __init__(self):
        self._parameters = []   # names of parameter attributes (strings)
        self._modules = []      # names of sub-module attributes (strings)
    
    def parameters(self):
        """Collect all learnable parameters (recursively)."""
        params = []
        for name in self._parameters:
            params.append(getattr(self, name))
        for name in self._modules:
            params.extend(getattr(self, name).parameters())
        return params
    
    def to(self, device):
        """Move all parameters to a device."""
        for name in self._parameters:
            setattr(self, name, getattr(self, name).to(device))
        for name in self._modules:
            getattr(self, name).to(device)
        return self
    
    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

### 3.1 Linear layer

Implement a `Linear` layer that stores weight $W$ and bias $b$ and calls your `linear()` function.

Initialize weights with the standard scheme: uniform in $\left[-\frac{1}{\sqrt{D_{\text{in}}}},\; \frac{1}{\sqrt{D_{\text{in}}}}\right]$.

In [9]:
class Linear(Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self._parameters = ['weight', 'bias']
        
        # YOUR CODE HERE
        # Initialize self.weight (out_dim, in_dim) and self.bias (out_dim,)
        # Use uniform initialization: k = 1/sqrt(in_dim), U(-k, k)
        raise NotImplementedError
    
    def forward(self, x, ctx=None):
        """Forward pass: delegate to your linear() function."""
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# --- Sanity check ---
layer = Linear(10, 5)
assert layer.weight.shape == (5, 10), f"Weight shape: {layer.weight.shape}"
assert layer.bias.shape == (5,), f"Bias shape: {layer.bias.shape}"
assert len(layer.parameters()) == 2

x_test = torch.randn(3, 10)
ctx = []
y_test = layer(x_test, ctx=ctx)
assert y_test.shape == (3, 5), f"Output shape: {y_test.shape}"
assert len(ctx) == 1, f"Expected 1 ctx entry, got {len(ctx)}"
print(f"✓ Linear layer works! Shape: (3, 10) → (3, 5)")

[Grand test: skipped execution of this cell for speed]


### 3.2 Models

Build two classifiers:

1. **SoftmaxClassifier** — a single linear layer (logits go directly to the loss function)
2. **MLP** — two linear layers with a ReLU in between: `Linear → ReLU → Linear`

In [10]:
class SoftmaxClassifier(Module):
    """Single linear layer: input → logits."""
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self._modules = ['fc']
        # YOUR CODE HERE
        raise NotImplementedError
    
    def forward(self, x, ctx=None):
        # YOUR CODE HERE
        raise NotImplementedError


class MLP(Module):
    """Two-layer MLP: input → hidden (ReLU) → logits."""
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self._modules = ['fc1', 'fc2']
        # YOUR CODE HERE
        raise NotImplementedError
    
    def forward(self, x, ctx=None):
        # YOUR CODE HERE  (remember to pass ctx to each operation!)
        raise NotImplementedError

In [ ]:
# --- Sanity check ---
softmax_model = SoftmaxClassifier(784, 10)
mlp_model = MLP(784, 128, 10)

x_test = torch.randn(4, 784)

ctx = []
y1 = softmax_model(x_test, ctx=ctx)
assert y1.shape == (4, 10), f"SoftmaxClassifier output: {y1.shape}"
print(f"✓ SoftmaxClassifier: (4, 784) → {y1.shape}, ctx has {len(ctx)} entries")

ctx = []
y2 = mlp_model(x_test, ctx=ctx)
assert y2.shape == (4, 10), f"MLP output: {y2.shape}"
print(f"✓ MLP: (4, 784) → {y2.shape}, ctx has {len(ctx)} entries")
print(f"  MLP parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")

[Grand test: skipped execution of this cell for speed]


---
## Part 4: SGD optimizer

Implement basic stochastic gradient descent: after each backward pass,
update every parameter with $\theta \leftarrow \theta - \eta \, \nabla_\theta$.

In [11]:
class SGD:
    def __init__(self, parameters, lr):
        """
        Args:
            parameters: list of tensors (from model.parameters())
            lr: learning rate
        """
        self.parameters = parameters
        self.lr = lr
    
    def zero_grad(self):
        """Reset all gradients to zero (or None)."""
        # YOUR CODE HERE
        raise NotImplementedError
    
    def step(self):
        """Update parameters: param -= lr * param.grad (in-place)."""
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# --- Sanity check ---
p = torch.tensor([1.0, 2.0, 3.0])
p.grad = torch.tensor([0.1, 0.2, 0.3])
opt = SGD([p], lr=1.0)
opt.step()
assert torch.allclose(p, torch.tensor([0.9, 1.8, 2.7])), f"After step: {p}"
opt.zero_grad()
assert p.grad is None or (p.grad == 0).all(), f"After zero_grad: {p.grad}"
print("✓ SGD works!")

[Grand test: skipped execution of this cell for speed]


---
## Part 5: Train on MNIST

Time to put it all together. We'll train your MLP on MNIST handwritten digits
using your custom autograd, then verify the results match PyTorch.

### Data loading

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data  = torchvision.datasets.MNIST('./data', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=256, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_data,  batch_size=1000, shuffle=False)

print(f"Training: {len(train_data)} images, Test: {len(test_data)} images")
print(f"Image shape: {train_data[0][0].shape} → flattened: {28*28}")

[Grand test: skipped execution of this cell for speed]


### Training and evaluation

Implement the training loop. For each batch:
1. Flatten images: `(B, 1, 28, 28)` → `(B, 784)`
2. Forward pass with `ctx`
3. Compute loss with `cross_entropy_loss`
4. Backward pass
5. Optimizer step

In [12]:
def train_epoch(model, optimizer, loader):
    """Train for one epoch. Returns average loss."""
    total_loss = 0
    n_batches = 0
    
    for images, labels in loader:
        # YOUR CODE HERE
        # 1. Flatten images
        # 2. Create ctx = []
        # 3. Zero gradients
        # 4. Forward pass (model + loss)
        # 5. Backward pass
        # 6. Optimizer step
        # 7. Track loss
        raise NotImplementedError
    
    return total_loss / n_batches


def evaluate(model, loader):
    """Compute accuracy (no gradients needed, so no ctx)."""
    correct = 0
    total = 0
    for images, labels in loader:
        x = images.view(images.size(0), -1)
        logits = model(x)           # no ctx → no recording
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total

### Train your MLP

In [ ]:
model = MLP(784, 128, 10)
optimizer = SGD(model.parameters(), lr=0.1)

history = {'loss': [], 'acc': []}

for epoch in range(5):
    loss = train_epoch(model, optimizer, train_loader)
    acc = evaluate(model, test_loader)
    history['loss'].append(loss)
    history['acc'].append(acc)
    print(f"Epoch {epoch+1}: loss = {loss:.4f}, test accuracy = {acc:.2%}")

[Grand test: skipped execution of this cell for speed]


In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, len(history['loss'])+1), history['loss'], 'o-')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(history['acc'])+1), [a*100 for a in history['acc']], 'o-', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_title('Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

[Grand test: skipped execution of this cell for speed]


In [ ]:
# Show some predictions
model_eval = model  # already trained
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    img, label = test_data[i]
    x = img.view(1, -1)
    logits = model_eval(x)
    pred = logits.argmax(dim=1).item()
    
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'pred: {pred}', color='green' if pred == label else 'red', fontweight='bold')
    axes[0, i].axis('off')
    
    # Show softmax probabilities as bar chart
    probs = torch.softmax(logits, dim=1).squeeze().detach()
    axes[1, i].bar(range(10), probs.numpy(), color=['green' if j == label else 'steelblue' for j in range(10)])
    axes[1, i].set_ylim(0, 1)
    axes[1, i].set_xticks(range(10))
    axes[1, i].tick_params(labelsize=7)

axes[0, 0].set_ylabel('Image')
axes[1, 0].set_ylabel('P(class)')
plt.suptitle('Sample predictions from your custom MLP', fontsize=13)
plt.tight_layout()
plt.show()

[Grand test: skipped execution of this cell for speed]


### Compare with PyTorch

Now train the exact same architecture using PyTorch's built-in `nn.Linear`, `F.cross_entropy`, and `torch.optim.SGD`.
The test accuracy should be very similar to your custom implementation.

In [13]:
import torch.nn as nn
import torch.optim as optim

class MLP_PyTorch(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model_pt = MLP_PyTorch()
optimizer_pt = optim.SGD(model_pt.parameters(), lr=0.1)

for epoch in range(5):
    model_pt.train()
    total_loss = 0
    for images, labels in train_loader:
        x = images.view(images.size(0), -1)
        optimizer_pt.zero_grad()
        loss = F.cross_entropy(model_pt(x), labels)
        loss.backward()
        optimizer_pt.step()
        total_loss += loss.item()
    
    model_pt.eval()
    correct = sum((model_pt(img.view(img.size(0), -1)).argmax(1) == lab).sum().item()
                  for img, lab in test_loader)
    acc = correct / len(test_data)
    print(f"Epoch {epoch+1}: loss = {total_loss/len(train_loader):.4f}, test accuracy = {acc:.2%}")

print("\nBoth should reach ~97%+ accuracy. If they do, your implementation is correct!")

NameError: name 'train_loader' is not defined

---
## Questions

1. Why do we accumulate gradients (`+=`) instead of assigning them (`=`)? Give an example of a computation where this matters.

2. What happens if you forget to call `optimizer.zero_grad()` before each forward pass? Try it and explain the behavior.

3. Our backward tape is a flat list (stack). PyTorch uses a DAG (directed acyclic graph) instead. What can a DAG handle that our list-based approach cannot?

*Your answers here.*

---
*Modern Computer Vision — Technion — Spring 2026*